In [1]:
import psycopg
from getpass import getpass

db_password = getpass("PostgreSQL password: ")

source_conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="source_db",
    user="vaez_etl",
    password=db_password
)

warehouse_conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="warehouse_db",
    user="vaez_etl",
    password=db_password
)

print("Connected to source_db and warehouse_db successfully.")

PostgreSQL password:  ········


Connected to source_db and warehouse_db successfully.


In [3]:
import pandas as pd

# Extract data from sourc_db 

extract_query = """
SELECT
    city_name,
    city_code,
    report_year,
    report_month,
    activity,
    status,
    work_order_count
FROM source_work_orders;
"""

with source_conn.cursor() as cursor:
    cursor.execute(extract_query)

    rows = cursor.fetchall()
    columns = [desc.name for desc in cursor.description]

source_df = pd.DataFrame(rows, columns=columns)

print(source_df.shape)
source_df.head()

(1536, 7)


,city_name,city_code,report_year,report_month,activity,status,work_order_count
0,شهر 1,6010,1401,7,test_1,in_progress,0
1,شهر 1,6010,1401,7,test_1,statement_preparation,0
2,شهر 1,6010,1401,7,test_1,at_headquarters,0
3,شهر 1,6010,1401,7,test_1,at_finance,0
4,شهر 1,6010,1401,7,test_2,in_progress,0


In [6]:
# Data Validation 
print("Rows:", len(source_df))
print("Null values:")
print(source_df.isnull().sum())

print("\nDuplicate business keys:")
print(
    source_df.duplicated(
        subset=[
            "city_code",
            "report_year",
            "report_month",
            "activity",
            "status"
        ]
    ).sum()
)

print("\nNegative counts:")
print((source_df["work_order_count"] < 0).sum())

Rows: 1536
Null values:
city_name           0
city_code           0
report_year         0
report_month        0
activity            0
status              0
work_order_count    0
dtype: int64

Duplicate business keys:
0

Negative counts:
0


In [7]:
# Trasfer Data 
cities = (
    source_df[["city_code", "city_name"]]
    .drop_duplicates()
    .sort_values("city_code")
)

dates = (
    source_df[["report_year", "report_month"]]
    .drop_duplicates()
    .sort_values(["report_year", "report_month"])
)

activities = (
    source_df[["activity"]]
    .drop_duplicates()
    .sort_values("activity")
)

statuses = (
    source_df[["status"]]
    .drop_duplicates()
    .sort_values("status")
)

print("Cities:", len(cities))
print("Dates:", len(dates))
print("Activities:", len(activities))
print("Statuses:", len(statuses))

Cities: 16
Dates: 3
Activities: 7
Statuses: 5


In [13]:
with warehouse_conn.cursor() as cursor:

    # -------------------------
    # dim_city
    # -------------------------
    for row in cities.itertuples(index=False):
        cursor.execute(
            """
            INSERT INTO dim_city (city_code, city_name)
            VALUES (%s, %s)
            ON CONFLICT (city_code)
            DO UPDATE SET
                city_name = EXCLUDED.city_name;
            """,
            (int(row.city_code), row.city_name)
        )

    # -------------------------
    # dim_date
    # -------------------------
    for row in dates.itertuples(index=False):
        cursor.execute(
            """
            INSERT INTO dim_date (report_year, report_month)
            VALUES (%s, %s)
            ON CONFLICT (report_year, report_month)
            DO NOTHING;
            """,
            (int(row.report_year), int(row.report_month))
        )

    # -------------------------
    # dim_activity
    # -------------------------
    for row in activities.itertuples(index=False):
        cursor.execute(
            """
            INSERT INTO dim_activity (activity_name)
            VALUES (%s)
            ON CONFLICT (activity_name)
            DO NOTHING;
            """,
            (row.activity,)
        )

    # -------------------------
    # dim_status
    # -------------------------
    for row in statuses.itertuples(index=False):
        cursor.execute(
            """
            INSERT INTO dim_status (status_name)
            VALUES (%s)
            ON CONFLICT (status_name)
            DO NOTHING;
            """,
            (row.status,)
        )

warehouse_conn.commit()

print("Dimensions loaded successfully.")

Dimensions loaded successfully.


In [16]:
with warehouse_conn.cursor() as cursor:

    for table in [
        "dim_city",
        "dim_date",
        "dim_activity",
        "dim_status"
    ]:
        cursor.execute(f"SELECT COUNT(*) FROM {table};")
        count = cursor.fetchone()[0]

        print(f"{table}: {count}")

dim_city: 16
dim_date: 3
dim_activity: 7
dim_status: 5


In [17]:
fact_insert_query = """
INSERT INTO fact_work_orders (
    city_id,
    date_id,
    activity_id,
    status_id,
    work_order_count
)
SELECT
    c.city_id,
    d.date_id,
    a.activity_id,
    s.status_id,
    src.work_order_count
FROM source_work_orders src

JOIN dim_city c
    ON src.city_code = c.city_code

JOIN dim_date d
    ON src.report_year = d.report_year
    AND src.report_month = d.report_month

JOIN dim_activity a
    ON src.activity = a.activity_name

JOIN dim_status s
    ON src.status = s.status_name;
"""

In [18]:
with warehouse_conn.cursor() as cursor:

    cursor.execute("""
        SELECT city_id, city_code
        FROM dim_city;
    """)
    city_map = {
        city_code: city_id
        for city_id, city_code in cursor.fetchall()
    }

    cursor.execute("""
        SELECT date_id, report_year, report_month
        FROM dim_date;
    """)
    date_map = {
        (year, month): date_id
        for date_id, year, month in cursor.fetchall()
    }

    cursor.execute("""
        SELECT activity_id, activity_name
        FROM dim_activity;
    """)
    activity_map = {
        name: activity_id
        for activity_id, name in cursor.fetchall()
    }

    cursor.execute("""
        SELECT status_id, status_name
        FROM dim_status;
    """)
    status_map = {
        name: status_id
        for status_id, name in cursor.fetchall()
    }

In [19]:
fact_records = []

for row in source_df.itertuples(index=False):

    fact_records.append(
        (
            city_map[row.city_code],
            date_map[(row.report_year, row.report_month)],
            activity_map[row.activity],
            status_map[row.status],
            row.work_order_count
        )
    )

print("Fact records:", len(fact_records))

Fact records: 1536


In [25]:
fact_query = """
INSERT INTO fact_work_orders (
    city_id,
    date_id,
    activity_id,
    status_id,
    work_order_count
)
VALUES (%s, %s, %s, %s, %s)

ON CONFLICT (
    city_id,
    date_id,
    activity_id,
    status_id
)
DO UPDATE SET
    work_order_count = EXCLUDED.work_order_count;
"""

try:
    with warehouse_conn.cursor() as cursor:
        cursor.executemany(
            fact_query,
            fact_records
        )

    warehouse_conn.commit()

    print(f"{len(fact_records)} fact records loaded successfully.")

except Exception as error:
    warehouse_conn.rollback()
    print("Fact loading failed:", error)

1536 fact records loaded successfully.


## DATA Validation

In [4]:
with warehouse_conn.cursor() as cursor:
    cursor.execute("SELECT COUNT(*) FROM fact_work_orders;")
    warehouse_count = cursor.fetchone()[0]

source_count = len(source_df)

print("Source rows:", source_count)
print("Warehouse fact rows:", warehouse_count)

if source_count == warehouse_count:
    print("PASS: Row counts match.")
else:
    print("FAIL: Row counts do not match.")

Source rows: 1536
Warehouse fact rows: 1536
PASS: Row counts match.


In [5]:
source_total = int(source_df["work_order_count"].sum())

with warehouse_conn.cursor() as cursor:
    cursor.execute("""
        SELECT SUM(work_order_count)
        FROM fact_work_orders;
    """)
    warehouse_total = cursor.fetchone()[0]

print("Source total:", source_total)
print("Warehouse total:", warehouse_total)

if source_total == warehouse_total:
    print("PASS: Totals match.")
else:
    print("FAIL: Totals do not match.")

Source total: 3875
Warehouse total: 3875
PASS: Totals match.


In [6]:
monthly_query = """
SELECT
    d.report_year,
    d.report_month,
    SUM(f.work_order_count) AS total_work_orders
FROM fact_work_orders f
JOIN dim_date d
    ON f.date_id = d.date_id
GROUP BY
    d.report_year,
    d.report_month
ORDER BY
    d.report_year,
    d.report_month;
"""

with warehouse_conn.cursor() as cursor:
    cursor.execute(monthly_query)
    results = cursor.fetchall()

for row in results:
    print(row)

(1401, 7, 1559)
(1401, 8, 1210)
(1401, 9, 1106)


In [7]:
check_query = """
SELECT
    c.city_name,
    c.city_code,
    d.report_year,
    d.report_month,
    a.activity_name,
    s.status_name,
    f.work_order_count
FROM fact_work_orders f

JOIN dim_city c
    ON f.city_id = c.city_id

JOIN dim_date d
    ON f.date_id = d.date_id

JOIN dim_activity a
    ON f.activity_id = a.activity_id

JOIN dim_status s
    ON f.status_id = s.status_id

ORDER BY
    d.report_year,
    d.report_month,
    c.city_code,
    a.activity_name,
    s.status_name

LIMIT 20;
"""

with warehouse_conn.cursor() as cursor:
    cursor.execute(check_query)
    rows = cursor.fetchall()

for row in rows:
    print(row)

('شهر 9', 2001, 1401, 7, 'test_1', 'at_finance', 0)
('شهر 9', 2001, 1401, 7, 'test_1', 'at_headquarters', 3)
('شهر 9', 2001, 1401, 7, 'test_1', 'in_progress', 8)
('شهر 9', 2001, 1401, 7, 'test_1', 'statement_preparation', 2)
('شهر 9', 2001, 1401, 7, 'test_2', 'at_finance', 0)
('شهر 9', 2001, 1401, 7, 'test_2', 'at_headquarters', 0)
('شهر 9', 2001, 1401, 7, 'test_2', 'in_progress', 0)
('شهر 9', 2001, 1401, 7, 'test_2', 'statement_preparation', 0)
('شهر 9', 2001, 1401, 7, 'test_3', 'at_finance', 0)
('شهر 9', 2001, 1401, 7, 'test_3', 'at_headquarters', 1)
('شهر 9', 2001, 1401, 7, 'test_3', 'in_progress', 0)
('شهر 9', 2001, 1401, 7, 'test_3', 'statement_preparation', 0)
('شهر 9', 2001, 1401, 7, 'test_4_a', 'at_consultant', 1)
('شهر 9', 2001, 1401, 7, 'test_4_a', 'at_finance', 1)
('شهر 9', 2001, 1401, 7, 'test_4_a', 'at_headquarters', 0)
('شهر 9', 2001, 1401, 7, 'test_4_a', 'in_progress', 27)
('شهر 9', 2001, 1401, 7, 'test_4_a', 'statement_preparation', 0)
('شهر 9', 2001, 1401, 7, 'test_4_b